# TTS Latency Horse-Race

A direct comparison of **Time to First Audio Byte (TTFAB)** across five TTS providers:
Neuphonic, Gradium, ElevenLabs, Cartesia, and OpenAI.

Results are most meaningful when the Codespace is launched in the **Europe West** region.

> **Before running:** copy `.env.example` → `.env` and fill in your five API keys.

In [2]:
import os, time, asyncio, json, uuid, inspect, websockets
from websockets.exceptions import ConnectionClosedOK
from dotenv import load_dotenv

load_dotenv()

_API_KEY_CACHE = {}

def get_api_key(name: str) -> str:
    if name in _API_KEY_CACHE:
        return _API_KEY_CACHE[name]
    v = os.getenv(name, "").strip()
    if not v:
        raise RuntimeError(f"Missing {name} — set it in your .env file")
    _API_KEY_CACHE[name] = v
    return v

async def maybe_await(x):
    return await x if inspect.isawaitable(x) else x

TEST_SENTENCES = [
    "The quick brown fox jumps over the lazy dog.",
    "A bright morning sun warmed the quiet village.",
    "Please send the final report before lunch today.",
    "The train arrived at the station five minutes early.",
    "She opened the window and listened to the rain.",
    "Our team finished the prototype ahead of schedule.",
    "The coffee shop on the corner was unusually busy.",
    "The meeting was moved to Thursday afternoon.",
    "He carefully placed the glass vase on the shelf.",
    "A small bird landed on the garden fence.",
    "They walked through the park after dinner.",
    "The new software update improved response time.",
    "I need to charge my phone before leaving.",
    "The children laughed as the kite rose higher.",
    "We packed sandwiches, fruit, and water for the trip.",
    "The blue notebook is inside the leather bag.",
    "Her voice sounded calm and confident on the call.",
    "The city lights reflected across the river.",
    "Please confirm your availability for tomorrow morning.",
    "The old wooden door creaked in the wind.",
]

ENABLED_ENGINES = ["openai", "neuphonic", "elevenlabs", "cartesia", "gradium"]

ENGINE_CONFIG = {
    "openai":     {"model": "gpt-4o-mini-tts", "voice": "coral"},
    "neuphonic":  {"voice_id": "fc854436-2dac-4d21-aa69-ae17b54e98eb", "sampling_rate": 24000, "lang_code": "en", "speed": 1.0},
    "elevenlabs": {"voice_id": "FGY2WhTYpPnrIDTdsKH5", "model_id": "eleven_flash_v2_5"},
    "cartesia":   {"voice_id": "78ab82d5-25be-4f7d-82b3-7ad64e5b85b2", "model_id": "sonic-3.5", "sample_rate": 44100},
    "gradium":    {"model_name": "default", "voice_id": "YTpq7expH9539ERJ", "output_format": "wav"},
}

async def openai_ttfab(text):
    from openai import AsyncOpenAI
    cfg = ENGINE_CONFIG["openai"]
    client = AsyncOpenAI(api_key=get_api_key("OPENAI_API_KEY"))
    start = time.perf_counter()
    async with client.audio.speech.with_streaming_response.create(
        model=cfg["model"], voice=cfg["voice"], input=text, response_format="pcm"
    ) as response:
        async for chunk in response.iter_bytes():
            if chunk: return time.perf_counter() - start
    raise RuntimeError("OpenAI returned no audio")

async def gradium_ttfab(text):
    import gradium
    cfg = ENGINE_CONFIG["gradium"]
    client = gradium.client.GradiumClient(api_key=get_api_key("GRADIUM_API_KEY"))
    start = time.perf_counter()
    stream = await client.tts_stream(setup=cfg, text=text)
    async for chunk in stream.iter_bytes():
        if chunk: return time.perf_counter() - start
    raise RuntimeError("Gradium returned no audio")

async def cartesia_ttfab(text):
    from cartesia import AsyncCartesia
    cfg = ENGINE_CONFIG["cartesia"]
    client = AsyncCartesia(api_key=get_api_key("CARTESIA_API_KEY"))
    async with client.tts.websocket_connect() as conn:
        start = time.perf_counter()
        await conn.send({
            "context_id": f"ctx_{uuid.uuid4().hex}",
            "model_id": cfg["model_id"],
            "transcript": text,
            "voice": {"mode": "id", "id": cfg["voice_id"]},
            "output_format": {"container": "raw", "encoding": "pcm_f32le", "sample_rate": cfg["sample_rate"]},
        })
        async for r in conn:
            if r.type == "chunk" and r.audio: return time.perf_counter() - start
            if getattr(r, "done", False): break
    raise RuntimeError("Cartesia returned no audio")

async def neuphonic_ttfab(text):
    from pyneuphonic import Neuphonic, TTSConfig, WebsocketEvents
    cfg = ENGINE_CONFIG["neuphonic"]
    ws = Neuphonic(api_key=get_api_key("NEUPHONIC_API_KEY")).tts.AsyncWebsocketClient()
    tts_config = TTSConfig(
        lang_code=cfg["lang_code"], voice_id=cfg["voice_id"],
        sampling_rate=cfg["sampling_rate"], speed=cfg["speed"], encoding="pcm_linear"
    )
    loop, start = asyncio.get_running_loop(), 0.0
    first_audio, stopped = loop.create_future(), loop.create_future()

    async def on_message(msg):
        nonlocal start
        if msg.data.audio and not first_audio.done(): first_audio.set_result(time.perf_counter() - start)
        if msg.data.stop and not stopped.done(): stopped.set_result(True)

    async def on_close(*_, **__):
        if not stopped.done(): stopped.set_result(True)

    ws.on(WebsocketEvents.MESSAGE, on_message)
    ws.on(WebsocketEvents.CLOSE, on_close)
    await maybe_await(ws.open(tts_config=tts_config))
    start = time.perf_counter()
    await maybe_await(ws.send(text, autocomplete=True))
    await asyncio.wait([first_audio, stopped], timeout=10, return_when=asyncio.FIRST_COMPLETED)
    await maybe_await(ws.close())
    if first_audio.done(): return first_audio.result()
    raise RuntimeError("Neuphonic returned no audio")

async def elevenlabs_ttfab(text):
    cfg = ENGINE_CONFIG["elevenlabs"]
    uri = f"wss://api.elevenlabs.io/v1/text-to-speech/{cfg['voice_id']}/stream-input?model_id={cfg['model_id']}&auto_mode=true"
    async with websockets.connect(uri) as ws:
        start = time.perf_counter()
        await ws.send(json.dumps({"xi_api_key": get_api_key("ELEVENLABS_API_KEY"), "text": " ",
            "voice_settings": {"stability": 0.5, "similarity_boost": 0.8},
            "generation_config": {"chunk_length_schedule": [120, 160, 250, 290]}}))
        await ws.send(json.dumps({"text": text + " ", "try_trigger_generation": True}))
        await ws.send(json.dumps({"text": ""}))
        try:
            while True:
                data = json.loads(await asyncio.wait_for(ws.recv(), timeout=10))
                if data.get("audio"): return time.perf_counter() - start
        except (asyncio.TimeoutError, ConnectionClosedOK):
            pass
    raise RuntimeError("ElevenLabs returned no audio")

TTS_FUNCS = {
    "openai":     openai_ttfab,
    "neuphonic":  neuphonic_ttfab,
    "elevenlabs": elevenlabs_ttfab,
    "cartesia":   cartesia_ttfab,
    "gradium":    gradium_ttfab,
}

async def main():
    results = []
    for sample_id, text in enumerate(TEST_SENTENCES, 1):
        for engine in ENABLED_ENGINES:
            try:
                ttfab = await TTS_FUNCS[engine](text)
                results.append({"engine": engine, "sample_id": sample_id, "sample_text": text, "ttfab_s": ttfab})
                print(f"✅ {engine} sample {sample_id}: TTFAB {ttfab:.3f}s")
            except Exception as exc:
                print(f"⚠️ {engine} sample {sample_id} failed: {exc}")
    print(f"\n✅ Finished – {len(results)} TTFAB measurements")
    return results

ModuleNotFoundError: No module named 'websockets'

In [ ]:
results = await main()

In [ ]:
import pandas as pd

pivot = (
    pd.DataFrame(results)[["engine", "ttfab_s", "sample_id"]]
    .pivot_table(index="sample_id", columns="engine", values="ttfab_s")
)

print("--- Mean TTFAB per engine (seconds) ---")
display(pivot.mean().to_frame("mean_ttfab_s").sort_values("mean_ttfab_s"))

pivot.boxplot(figsize=(10, 5))
